# Aula 4: Features

Criação do notebook "Features".


## Quantidade de Produtos Distintos

Vamos encontar quais são os produtos mais transacionados e criar suas categorias.
Usaremos toda a base até a janela de corte '2026-06-01'. 

In [0]:
%sql
WITH
tb_transacoes AS (
    SELECT 
        t3.DescNomeProduto,
        COUNT(t1.IdTransacao) AS QtdeTransacoes
    FROM workspace.tmw_loyalty.transacoes AS t1

    LEFT JOIN workspace.tmw_loyalty.transacao_produto AS t2
        ON t1.IdTransacao = t2.IdTransacao
    
    LEFT JOIN workspace.tmw_loyalty.produtos AS t3
        ON t2.IdProduto = t3.IdProduto
    
    WHERE t1.DtCriacao < '2026-06-01' --AND t1.DtCriacao >= '2026-06-01' - interval 28 days
    GROUP BY ALL
) 

SELECT 
    DescNomeProduto,
    QtdeTransacoes,
    ROUND(QtdeTransacoes / SUM(QtdeTransacoes) OVER () , 4) AS pct
FROM tb_transacoes
ORDER BY pct DESC

Criaria então 6 grupos:
1. 'ChatMessage'
2. 'Lista de presença'
3. 'Presença Streak'
4. 'Resgatar Ponei'
5. 'Troca de Pontos StreamElements'
6. Outros


## Features: PONTOS

Vai dar origem a `fs_pontos.sql`.

Cou parametrizar a query para recever `{date}` quando `'2026-06-01'` 

In [0]:
%sql
WITH tb_transacoes AS (
    SELECT *
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01' AND DtCriacao >= '2026-06-01' - interval 28 days
),

tb_cliente_agg AS (

    SELECT IdCliente,
        COUNT(distinct date(DtCriacao)) AS QtdeFrequencia,
        SUM(QtdePontos) AS QtdePontos,
        SUM(CASE WHEN QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS QtdePontosPositivos,
        MIN(date_diff('2026-06-01', DtCriacao)) AS Recencia,
        COUNT(idTransacao) AS QtdeTransacoes
    FROM tb_transacoes

    GROUP BY ALL
),

tb_cliente_produto AS (
    SELECT 
        t1.IdCliente,
        COUNT(t1.idTransacao) AS QtdeTransacoes,
        -- Percentual de transações por produto
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'ChatMessage' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_chat,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Lista de presença' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_presenca,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_streak,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Resgatar Ponei' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_ponei,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Troca de Pontos StreamElements' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_streamElements,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto NOT IN ('ChatMessage', 'Lista de presença', 'Presença Streak', 'Resgatar Ponei', 'Troca de Pontos StreamElements') OR t3.DescNomeProduto IS NULL THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_outros,
        -- Percentual de pontos por produto
        SUM(CASE WHEN t3.DescNomeProduto = 'ChatMessage' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_chat,
        SUM(CASE WHEN t3.DescNomeProduto = 'Lista de presença' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_presenca,
        SUM(CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_streak,
        SUM(CASE WHEN t3.DescNomeProduto = 'Resgatar Ponei' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_ponei,
        SUM(CASE WHEN t3.DescNomeProduto = 'Troca de Pontos StreamElements' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_streamElements,
        SUM(CASE WHEN t3.DescNomeProduto NOT IN ('ChatMessage', 'Lista de presença', 'Presença Streak', 'Resgatar Ponei', 'Troca de Pontos StreamElements') OR t3.DescNomeProduto IS NULL THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_outros,
        -- Flag de Streak
        MAX( CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN 1 ELSE 0 END) AS flStreak,
        -- Dias desde o último Streak
        MIN(CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN date_diff('2026-06-01', t1.DtCriacao) ELSE NULL END) AS diasDesdeUltimoStreak,
        -- Quantidade de produto distinto
        COUNT(DISTINCT t2.IdProduto) AS qtdeProdutoDistintos

    FROM tb_transacoes AS t1

    LEFT JOIN workspace.tmw_loyalty.transacao_produto AS t2
        ON t1.IdTransacao = t2.IdTransacao
    
    LEFT JOIN workspace.tmw_loyalty.produtos AS t3
        ON t2.IdProduto = t3.IdProduto
    
    WHERE t1.DtCriacao < '2026-06-01' AND t1.DtCriacao >= '2026-06-01' - interval 28 days
    GROUP BY ALL
),

tb_vida AS (
    SELECT
        t1.idCliente,
        MAX(date_diff('2026-06-01', t1.dtCriacao)) AS diasPrimeiraTransacao,
        COUNT(DISTINCT DATE(DtCriacao)) AS freqVida,
        SUM(t1.QtdePontos) AS SaldoPontos
    FROM workspace.tmw_loyalty.transacoes AS t1
    WHERE DtCriacao < '2026-06-01'
    GROUP BY ALL
),

tb_join AS (
    SELECT 
        t1.*,
        (t1.QtdePontosPositivos - (SELECT AVG(QtdePontosPositivos) FROM tb_cliente_agg) ) / (SELECT stddev(QtdePontosPositivos) FROM tb_cliente_agg) AS zScore,
        t3.diasPrimeiraTransacao,
        t3.freqVida,
        t3.SaldoPontos,
        t2.pctTransacao_chat,
        t2.pctTransacao_presenca,
        t2.pctTransacao_streak,
        t2.pctTransacao_ponei,
        t2.pctTransacao_streamElements,
        t2.pctTransacao_outros,
        t2.pctPontosAbs_chat,
        t2.pctPontosAbs_presenca,
        t2.pctPontosAbs_streak,
        t2.pctPontosAbs_ponei,
        t2.pctPontosAbs_streamElements,
        t2.pctPontosAbs_outros,
        t2.flStreak,
        t2.diasDesdeUltimoStreak,
        t2.qtdeProdutoDistintos


    FROM tb_cliente_agg AS t1

    LEFT JOIN tb_cliente_produto AS t2
        ON t1.idCliente = t2.idCliente

    LEFT JOIN tb_vida AS t3
        ON t1.idCliente = t3.idCliente
)

SELECT
    '2026-06-01' AS dtRef,
    *
FROM tb_join



## Features: CURSOS

Vai dar origem a `fs_cursos.sql`